# Oppimisprojekti 3, osa 2: puheentunnistus Whisper-mallilla

Tässä notebookissa käytetään Hugging Facen `transformers`-kirjaston Whisper-malleja suomenkielisen puheen tunnistamiseen. Notebook lukee itse nauhoitetut alle 30 sekunnin äänitteet kansiosta `audio_samples` ja ajaa vertailun usealla mallikoolla.

**Tekoälyn käyttö:** Tekoälyä käytettiin apuna notebookin rakenteen suunnittelussa, koodin korjaamisessa ja raporttipohjan muotoilussa. Ääninäytteet on nauhoitettu itse, Whisper-ajot tehdään paikallisesti, ja lopullinen arviointi kirjoitetaan omien ajotulosten perusteella.


## Mel-spektrogrammi omin sanoin

Whisper ei syötä neuroverkolle raakaa ääniaaltoa. Ääni muunnetaan ensin mel-spektrogrammiksi.

- **x-akseli** kuvaa aikaa: vasemmalta oikealle edetään äänitteen alusta loppuun.
- **y-akseli** kuvaa taajuuksia mel-asteikolla. Mel-asteikko painottaa taajuuksia ihmiskuulon kannalta mielekkäämmin kuin lineaarinen hertsiasteikko.
- **väri** kuvaa voimakkuutta desibeleinä. Kirkkaammat tai voimakkaammat värit tarkoittavat, että kyseisellä ajanhetkellä ja taajuusalueella on enemmän energiaa.

Tämä esitysmuoto sopii neuroverkolle paremmin kuin raaka aalto, koska puheen olennaiset piirteet, kuten vokaalit, konsonantit ja äänenpainot, näkyvät paikallisina kuvioina ajan ja taajuuden tasossa. Transformer saa siis järjestetyn piirrejonon, jota se voi käsitellä samaan tapaan kuin osassa 1 tokenijonoa.

In [ ]:
# Tarvittaessa asenna riippuvuudet esimerkiksi:
# %pip install torch transformers librosa soundfile matplotlib pandas jiwer accelerate

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Laite:", device)

## Ääninäytteet

Notebook lukee äänitiedostot automaattisesti kansiosta `audio_samples`. Tässä projektissa käytössä on useita lyhyitä itse nauhoitettuja näytteitä: selkeä puhe, taustahäly, nopeampi tai epäselvempi puhe, murteellinen tai puhekielinen puhe sekä englanninkielisiä vertailunäytteitä.

Kaikki nykyiset `.wav`-tiedostot ovat tarkistuksen perusteella 16 kHz mono -ääntä ja alle 30 sekuntia pitkiä, joten ne sopivat hyvin Whisperin kokeiluun.

Koska tarkat referenssitekstit on kirjoitettu tiedostoon `audio_samples/referenssitekstit.csv`, notebook voi laskea myös WER-virhemittarin ja tukea laadullista arviota numeerisella vertailulla.


In [ ]:
AUDIO_DIR = Path("audio_samples")
AUDIO_DIR.mkdir(exist_ok=True)
REFERENCE_PATH = AUDIO_DIR / "referenssitekstit.csv"

sample_descriptions = {
    "selkea_puhe.wav": {
        "kuvaus": "Selkeä suomenkielinen puhe, rauhallinen lukutapa.",
        "oikea_teksti": "",
    },
    "taustahaly.wav": {
        "kuvaus": "Suomenkielinen puhe taustahälyn kanssa.",
        "oikea_teksti": "",
    },
    "aaninayte_nopea.wav": {
        "kuvaus": "Nopeampi tai hieman epäselvempi suomenkielinen puhe.",
        "oikea_teksti": "",
    },
    "murre_aaninayte.wav": {
        "kuvaus": "Puhekielinen tai murteellinen suomenkielinen näyte.",
        "oikea_teksti": "",
    },
    "murre_aaninayte2.wav": {"kuvaus": "Toinen puhekielinen tai murteellinen suomenkielinen näyte.", "oikea_teksti": ""},
    "english_sample1.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample2.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample3.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample_fast.wav": {
        "kuvaus": "Nopea englanninkielinen vertailunäyte.",
        "oikea_teksti": "",
    },
    "selkea_puhe_1.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 1.", "oikea_teksti": ""},
    "selkea_puhe_2.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 2.", "oikea_teksti": ""},
    "selkea_puhe_3.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 3.", "oikea_teksti": ""},
    "selkea_puhe_4.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 4.", "oikea_teksti": ""},
    "selkea_puhe_5.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 5.", "oikea_teksti": ""},
}

if REFERENCE_PATH.exists():
    references_df = pd.read_csv(REFERENCE_PATH, encoding="utf-8")
    references_df = references_df.fillna("")
    for row in references_df.to_dict(orient="records"):
        filename = str(row.get("tiedosto", "")).strip()
        if not filename:
            continue
        sample_descriptions.setdefault(filename, {"kuvaus": "Kuvaus puuttuu", "oikea_teksti": ""})
        sample_descriptions[filename]["oikea_teksti"] = str(row.get("oikea_teksti", "")).strip()
        comment = str(row.get("kommentti", "")).strip()
        if comment and sample_descriptions[filename].get("kuvaus") == "Kuvaus puuttuu":
            sample_descriptions[filename]["kuvaus"] = comment
    filled_refs = sum(bool(item.get("oikea_teksti", "").strip()) for item in sample_descriptions.values())
    print(f"Luettiin {filled_refs} referenssitekstiä tiedostosta {REFERENCE_PATH.name}")
else:
    print(f"Referenssitiedostoa ei löytynyt: {REFERENCE_PATH}")

supported = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in supported])

print(f"Löytyi {len(audio_files)} äänitiedostoa kansiosta {AUDIO_DIR.resolve()}")
for p in audio_files:
    print("-", p.name)


In [ ]:
def load_audio(path, target_sr=16000):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)
    if duration > 30:
        print(f"Varoitus: {path.name} on {duration:.1f} s. Whisperin peruskokeeseen suositellaan alle 30 s pätkiä.")
    return y, sr, duration

for path in audio_files:
    y, sr, duration = load_audio(path)
    desc = sample_descriptions.get(path.name, {}).get("kuvaus", "Kuvaus puuttuu")
    print(f"{path.name}: {duration:.2f} s, {sr} Hz, {desc}")
    display(Audio(y, rate=sr))

## Mel-spektrogrammin visualisointi

Aja seuraava solu yhdelle tai useammalle näytteelle ja liitä havaintosi raporttiin. Kiinnitä huomiota siihen, missä kohtaa puhe, tauot ja mahdollinen taustahäly näkyvät.

In [ ]:
def plot_mel_spectrogram(path, n_mels=80):
    y, sr, duration = load_audio(path)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=160, n_fft=400)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Aaltomuoto: {path.name}")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spektrogrammi")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_mel_spectrogram(audio_files[0])
else:
    print("Lisää äänitiedostoja audio_samples-kansioon ja aja solu uudelleen.")

## Whisper-mallien vertailu

Oletuksena vertaillaan malleja `tiny` ja `base`, koska ne latautuvat ja ajavat kohtuullisen nopeasti. Lisää listaan esimerkiksi `small` tai `large-v3`, jos käytössä on tarpeeksi muistia ja aikaa.

In [ ]:
MODEL_SIZES = ["tiny", "base"]  # voit kokeilla myös: "small", "medium", "large-v3"
LANGUAGE = "fi"
TASK = "transcribe"
MAX_SAMPLES = None  # vaihda arvoksi 1, jos haluat ensin nopean testiajon

def model_id(size):
    return "openai/whisper-large-v3" if size == "large-v3" else f"openai/whisper-{size}"


def load_whisper(size):
    name = model_id(size)
    print(f"Ladataan malli {name}...", flush=True)
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to(device)
    model.eval()
    print(f"Malli {name} ladattu.", flush=True)
    return processor, model


def transcribe(path, processor, model, language=LANGUAGE, task=TASK):
    y, sr, duration = load_audio(path)
    inputs = processor(y, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, language=language, task=task)
    seconds = time.perf_counter() - start

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text, duration, seconds


In [ ]:
results = []
files_to_run = audio_files[:MAX_SAMPLES] if MAX_SAMPLES else audio_files

print(f"Ajettavia tiedostoja: {len(files_to_run)}", flush=True)
print(f"Mallit: {', '.join(MODEL_SIZES)}", flush=True)

for size in MODEL_SIZES:
    print()
    print(f"Aloitetaan Whisper {size}...", flush=True)
    processor, whisper_model = load_whisper(size)

    for index, path in enumerate(files_to_run, start=1):
        print(f"  [{index}/{len(files_to_run)}] Tunnistetaan: {path.name}", flush=True)
        language = "en" if path.name.startswith("english") else "fi"
        prediction, duration, seconds = transcribe(path, processor, whisper_model, language=language)
        ref = sample_descriptions.get(path.name, {}).get("oikea_teksti", "")
        results.append({
            "malli": size,
            "tiedosto": path.name,
            "kieli": language,
            "kesto_s": round(duration, 2),
            "aika_s": round(seconds, 2),
            "reaaliaikakerroin": round(seconds / duration, 2) if duration else np.nan,
            "oikea_teksti": ref,
            "tunnistus": prediction,
        })
        print(f"    Valmis {seconds:.2f} sekunnissa", flush=True)

    del whisper_model
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv("osa2_whisper_tulokset_oma_ajo.csv", index=False, encoding="utf-8-sig")
display(results_df)
print("Tulokset tallennettu tiedostoon osa2_whisper_tulokset_oma_ajo.csv", flush=True)


In [ ]:
# WER-laskenta, kun tarkat referenssit on kirjoitettu CSV-tiedostoon.
# Mittari ei kerro kaikkea puheentunnistuksen laadusta, mutta se antaa
# hyödyllisen numeerisen vertailun mallien välillä.

try:
    from jiwer import wer
    if not results_df.empty and results_df["oikea_teksti"].fillna("").str.len().gt(0).any():
        results_df["WER"] = results_df.apply(
            lambda r: wer(r["oikea_teksti"], r["tunnistus"]) if r["oikea_teksti"] else np.nan,
            axis=1,
        )
        display(results_df[["malli", "tiedosto", "kieli", "aika_s", "reaaliaikakerroin", "WER", "tunnistus"]])
    else:
        print("WER-laskentaa ei ajettu, koska tarkkoja referenssitekstejä ei ole täytetty.")
except Exception as exc:
    print("WER-laskentaa ei ajettu:", exc)


## Raportointitaulukko ja tulosten analyysi

### Käytetyt ääninäytteet

Aineistossa on yhteensä 14 lyhyttä ääninäytettä. Mukana on selkeää suomenkielistä puhetta, viisi pidempää lisänäytettä eri puhujilta, nopeampi ja hieman epäselvempi suomenkielinen näyte, kaksi murteellista tai puhekielistä näytettä, yksi taustahälyinen näyte sekä neljä englanninkielistä vertailunäytettä. Näin kokeessa näkyvät sekä mallin perussuorituskyky että vaikeammat tilanteet, joissa ääntäminen, puhenopeus, murre tai taustamelu voivat heikentää tunnistusta.

### Vertailu eri mallien välillä

Whisper `tiny` oli selvästi nopeampi kuin `base`. Oman ajon perusteella `tiny`-mallin keskimääräinen reaaliaikakerroin oli noin 0,78, kun `base`-mallilla vastaava arvo oli noin 1,25. Tämä tarkoittaa, että `tiny` käsitteli näytteet keskimäärin selvästi nopeammin, ja useissa lyhyissä näytteissä se toimi lähellä reaaliaikaa tai sitä nopeammin.

Tarkkuudessa `base` oli kuitenkin parempi. Kun referenssitekstien avulla laskettiin WER, `tiny`-mallin keskimääräinen virhesuhde oli noin 0,59 ja `base`-mallin noin 0,41. Ero näkyi erityisesti pidemmissä suomenkielisissä näytteissä ja englanninkielisissä vertailuissa: `base` säilytti lauserakenteen paremmin, jätti vähemmän sanoja pois ja tuotti useammin kokonaisia järkeviä virkkeitä.

### Missä malli onnistui ja epäonnistui

Molemmat mallit tunnistivat parhaiten selkeän ja rauhallisesti puhutun äänen. Englanninkielisissä näytteissä etenkin `base` suoriutui erittäin hyvin: esimerkiksi `english_sample1.wav` ja `english_sample2.wav` tunnistuivat lähes oikein sanasta sanaan. Tämä osoittaa, että esikoulutettu Whisper hyödyntää vahvaa monikielistä pohjaa myös silloin, kun projektin painopiste on suomenkielisessä puheessa.

Vaikeimmat näytteet olivat murteelliset puheotteet ja osin myös taustahälyinen puhe. Näissä molemmat mallit tekivät paljon äänne- ja sanatasoisia virheitä, mutta `tiny` hajosi selvemmin ja tuotti paikoin hyvin epätarkkaa tekstiä. Erityisen hankala oli `murre_aaninayte.wav`, jossa molemmat mallit jäivät kauas referenssistä. Myös `selkea_puhe_2.wav` osoitti, että pienempi malli voi joskus menettää pitkän lauseen rakenteen lähes kokonaan, vaikka puhe ei itsessään ole erityisen vaikeaa.

### Milloin pienempi malli riittää

Pienempi `tiny`-malli riittää tilanteissa, joissa halutaan nopea karkea litterointi selkeästä lyhyestä puheesta eikä pieniä virheitä pidetä kriittisinä. Se voi sopia esimerkiksi kokeiluihin, käyttöliittymäprototyyppeihin tai tilanteisiin, joissa vasteaika on tärkeämpi kuin tarkka sanamuoto. Kun taas tarvitaan parempaa tarkkuutta, pidempiä yhtenäisiä lauseita tai enemmän kestävyyttä eri puhujien ja vaikeampien näytteiden kanssa, `base` on tässä aineistossa perustellusti parempi valinta.


In [ ]:
if "results_df" not in globals() or results_df.empty:
    print("Aja ensin Whisper-vertailusolu, jotta results_df muodostuu.")
else:
    comparison = (
        results_df
        .pivot_table(
            index=["tiedosto", "kieli", "kesto_s"],
            columns="malli",
            values=["aika_s", "reaaliaikakerroin", "tunnistus"],
            aggfunc="first",
        )
        .sort_index()
    )

    averages = (
        results_df
        .groupby("malli", as_index=False)
        .agg(
            naytteita=("tiedosto", "count"),
            keskim_aika_s=("aika_s", "mean"),
            keskim_reaaliaikakerroin=("reaaliaikakerroin", "mean"),
        )
        .round(2)
    )

    display(comparison)
    display(averages)

    if "WER" in results_df.columns:
        quality = (
            results_df
            .groupby(["malli", "kieli"], as_index=False)
            .agg(keskiarvo_WER=("WER", "mean"))
            .round(3)
        )
        display(quality)


### Analyysi oman ajon jälkeen

Oman ajon perusteella `base` oli tämän aineiston kokonaisuutena parempi malli, vaikka se oli hitaampi. Se tuotti keskimäärin tarkempaa tekstiä sekä suomeksi että englanniksi, ja ero näkyi erityisesti silloin, kun lauseet olivat pitkiä tai puheessa oli vaihtelua. `tiny` oli hyödyllinen nopeuden vuoksi, mutta tunnistus hajosi helpommin tavurakenteisiin, vääristyneisiin sanoihin ja pois jääviin osiin.

Taustahäly, nopea puhe ja murteellisuus heikensivät kaikkien mallien tuloksia, mikä näkyi erityisesti suomalaisissa murresanoissa ja sananrajoissa. Silti tuloksissa on selvä looginen ero mallikokojen välillä: suurempi malli sietää paremmin vaihtelua ja säilyttää useammin lauseen merkityksen, vaikka jokainen sana ei osuisikaan täysin oikein. Tästä syystä kokeen päätelmä on, että `tiny` sopii nopeaan kokeiluun, mutta `base` on käytännöllisempi, kun tavoitteena on käyttökelpoisempi litterointi.

### Miksi Whisper käyttää mel-spektrogrammia?

Mel-spektrogrammi tiivistää puhesignaalin sellaiseen muotoon, jossa ajan mukana muuttuvat taajuuspiirteet näkyvät selkeinä rakenteina. Tämä helpottaa neuroverkon oppimista, koska raakaaaltomuoto sisältää paljon yksityiskohtaista vaihtelua, joka ei sellaisenaan ole mallille yhtä informatiivista. Mel-asteikko painottaa ihmiskuulon kannalta olennaisia taajuuksia, joten malli saa syötteeksi tiiviimmän ja merkityksellisemmän esityksen puheesta.

### Yhteys osaan 1

Osassa 1 transformer käsitteli tekstin tokenijonoa ja ennusti seuraavaa tokenia. Whisperissa syöte ei ole tekstiä vaan ääntä, joka muunnetaan ensin mel-spektrogrammiksi eli aika-taajuuspiirteiden jonoksi. Molemmissa tapauksissa transformer hyödyntää attention-mekanismia järjestetyn syötteen riippuvuuksien mallintamiseen. Siksi sama perusarkkitehtuuri toimii sekä tekstigeneraatiossa että puheentunnistuksessa, vaikka syötteen esitysmuoto on eri.
